# Reject Policy Demo

This notebook demonstrates the **reject policy system** in calibrated_explanations.

The reject policy system allows you to control how the runtime handles uncertain predictions:
- **NONE**: Legacy behavior (no reject orchestration)
- **FLAG**: Process all instances, include rejection status in results
- **ONLY_REJECTED**: Process only rejected (uncertain) instances
- **ONLY_ACCEPTED**: Process only accepted (confident) instances

At the end, we also show the legacy `predict_reject` workflow (policy-first example preferred).

## Deprecated Policies

The following policy names are deprecated and will be removed in v1.0.0:
- `PREDICT_AND_FLAG` → use `FLAG`
- `EXPLAIN_ALL` → use `FLAG`
- `EXPLAIN_REJECTS` → use `ONLY_REJECTED`
- `EXPLAIN_NON_REJECTS` → use `ONLY_ACCEPTED`
- `SKIP_ON_REJECT` → use `ONLY_ACCEPTED`

In [1]:
from __future__ import annotations

import warnings

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)


from calibrated_explanations import __version__

print(f"Calibrated Explanations version: {__version__}")

Calibrated Explanations version: v0.11.0


In [2]:
# Data and model
X, y = load_breast_cancer(return_X_y=True)
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
x_proper, x_cal, y_proper, y_cal = train_test_split(
    x_train, y_train, test_size=0.25, random_state=42
)

# Use WrapCalibratedExplainer consistently
from calibrated_explanations import WrapCalibratedExplainer

model = RandomForestClassifier(n_estimators=10, random_state=42)
wrapper = WrapCalibratedExplainer(model)
wrapper.fit(x_proper, y_proper)
wrapper.calibrate(x_cal, y_cal)

# Initialize reject learner for later use
# wrapper.initialize_reject_learner()
print("Wrapper calibrated and reject learner initialized")

Wrapper calibrated and reject learner initialized


In [3]:
# Select 10 instances covering the range from robust to fragile
# Fragility is estimated based on calibrated probability distance from decision boundary (0.5)
# Instances near 0.5 are fragile (novel), instances near 0 or 1 are robust (confident) or ambiguous.

# Get calibrated probabilities for all test instances
proba = wrapper.predict_proba(x_test)
# Distance from decision boundary - smaller = more fragile
fragility_score = np.abs(proba[:, 1] - 0.5)

# Sort by fragility (most fragile first)
sorted_indices = np.argsort(fragility_score)

# Select 10 instances spanning the fragility range
# Pick from different quantiles to get good coverage
n_select = 10
quantile_indices = np.linspace(0, len(sorted_indices) - 1, n_select, dtype=int)
selected_indices = sorted_indices[quantile_indices]

# Create the filtered test set
X_demo = x_test[selected_indices]
y_demo = y_test[selected_indices]
proba_demo = proba[selected_indices]

print("Selected 10 instances from most fragile to most robust:")
print("-" * 60)
for i, (idx, p, frag) in enumerate(zip(selected_indices, proba_demo, fragility_score[selected_indices], strict=False)):
    status = "FRAGILE" if frag < 0.2 else ("MODERATE" if frag < 0.4 else "ROBUST")
    print(f"  [{i}] Original idx={idx:3d}, P(class=1)={p[1]:.3f}, distance_from_0.5={frag:.3f} -> {status}")

# Verify rejection status at different confidence levels using policy-first envelope
from calibrated_explanations.core.reject.policy import RejectPolicy

print("\nRejection status at different confidence levels (policy-first):")
for conf in [0.5, 0.90, 0.95, 0.99]:
    res = wrapper.predict(X_demo, reject_policy=RejectPolicy.FLAG, confidence=conf)
    meta = res.metadata or {}
    n_rejected = int(np.sum(res.rejected)) if res.rejected is not None else 0
    n_amb = int(np.sum(meta.get('ambiguity_mask', np.zeros(len(X_demo), dtype=bool))))
    n_novel = int(np.sum(meta.get('novelty_mask', np.zeros(len(X_demo), dtype=bool))))
    print(f"  conf={conf}: rejected={n_rejected}/10 ({meta.get('reject_rate', 0.0):.1%}), ambiguity={n_amb}/10 ({meta.get('ambiguity_rate', 0.0):.1%}), novelty={n_novel}/10 ({meta.get('novelty_rate', 0.0):.1%})")

Selected 10 instances from most fragile to most robust:
------------------------------------------------------------
  [0] Original idx=108, P(class=1)=0.517, distance_from_0.5=0.017 -> FRAGILE
  [1] Original idx= 45, P(class=1)=0.860, distance_from_0.5=0.360 -> MODERATE
  [2] Original idx= 49, P(class=1)=0.860, distance_from_0.5=0.360 -> MODERATE
  [3] Original idx= 34, P(class=1)=0.960, distance_from_0.5=0.460 -> ROBUST
  [4] Original idx= 42, P(class=1)=0.960, distance_from_0.5=0.460 -> ROBUST
  [5] Original idx=  4, P(class=1)=0.960, distance_from_0.5=0.460 -> ROBUST
  [6] Original idx=105, P(class=1)=0.960, distance_from_0.5=0.460 -> ROBUST
  [7] Original idx=  1, P(class=1)=0.026, distance_from_0.5=0.474 -> ROBUST
  [8] Original idx= 71, P(class=1)=0.026, distance_from_0.5=0.474 -> ROBUST
  [9] Original idx=113, P(class=1)=0.026, distance_from_0.5=0.474 -> ROBUST

Rejection status at different confidence levels (policy-first):
  conf=0.5: rejected=3/10 (30.0%), ambiguity=0/10 (0.

In [4]:
# Helper summarizer tuned for explain_factual outputs
def _summarize_result(result, *, label: str):
    is_reject_result = hasattr(result, "policy") and hasattr(result, "rejected")
    print("\n---", label, "---")
    print("type:", type(result))
    # Preserve legacy printing for non-reject results
    if not is_reject_result:
        if hasattr(result, "__len__") and not isinstance(result, (str, bytes, dict)):
            try:
                print("len:", len(result))
                expls = getattr(result, "explanations", None)
                if expls is not None:
                    total = len(expls)
                    explained = sum(1 for e in expls if e is not None)
                    print(f"explained_count: {explained} / {total}")
                else:
                    try:
                        explained = sum(1 for e in result if e is not None)
                        print(f"explained_count (iter): {explained} / {len(result)}")
                    except Exception:
                        pass
            except Exception:
                pass
        else:
            print(repr(result))
        return
    # RejectResult-specific summary
    print("policy:", result.policy)
    rejected = getattr(result, "rejected", None)
    if rejected is not None:
        rejected = np.asarray(rejected, dtype=bool)
        print("rejected_count:", int(rejected.sum()), "/", int(rejected.size))
    pred = getattr(result, "prediction", None)
    print("has_prediction:", pred is not None)
    expl = getattr(result, "explanation", None)
    if expl is None:
        print("explanation: None")
    else:
        if hasattr(expl, "__len__") and not isinstance(expl, (str, bytes, dict)):
            print("explanations_len:", len(expl))
            
            # Build index mapping based on policy
            # For ONLY_REJECTED: explanation indices map to rejected instance indices
            # For ONLY_ACCEPTED: explanation indices map to accepted instance indices
            # For FLAG: explanation indices map 1:1 with instance indices
            policy = result.policy
            if rejected is not None:
                if policy is RejectPolicy.ONLY_REJECTED:
                    # Explanations are only for rejected instances
                    original_indices = [i for i, r in enumerate(rejected) if r]
                elif policy is RejectPolicy.ONLY_ACCEPTED:
                    # Explanations are only for accepted instances
                    original_indices = [i for i, r in enumerate(rejected) if not r]
                else:
                    # FLAG or other: 1:1 mapping
                    original_indices = list(range(len(rejected)))
            else:
                original_indices = list(range(len(expl)))
            
            for expl_idx, e in enumerate(expl):
                # Map explanation index to original instance index
                if expl_idx < len(original_indices):
                    orig_idx = original_indices[expl_idx]
                    if rejected is not None and orig_idx < len(rejected):
                        status = "REJECTED" if bool(rejected[orig_idx]) else "ACCEPTED"
                    else:
                        status = "UNKNOWN"
                else:
                    status = "UNKNOWN"
                    orig_idx = expl_idx
                
                if e is None:
                    print(f" [{expl_idx}] (orig={orig_idx}) {status} - None")
                    continue
                if hasattr(e, "prediction"):
                    try:
                        p = e.prediction
                        if isinstance(p, dict) and "predict" in p:
                            print(f' [{expl_idx}] (orig={orig_idx}) {status} - predict={p["predict"]:.3f}')
                        else:
                            print(f" [{expl_idx}] (orig={orig_idx}) {status} - prediction_present")
                    except Exception:
                        print(f" [{expl_idx}] (orig={orig_idx}) {status} - prediction (unprintable)")
                else:
                    print(f" [{expl_idx}] (orig={orig_idx}) {status} - explanation_type={type(e)}")
        else:
            print("explanation_type:", type(expl))

In [5]:
from calibrated_explanations.core.reject.policy import RejectPolicy

POLICIES = [
    RejectPolicy.NONE,
    RejectPolicy.FLAG,
    RejectPolicy.ONLY_REJECTED,
    RejectPolicy.ONLY_ACCEPTED,
]

# Demonstrate each policy with explain_factual using our 10 selected instances
# NOTE: reject_policy requires the plugin system (_use_plugin=True, the default)
print("=" * 70)
print("REJECT POLICY DEMONSTRATION (10 instances: fragile -> robust)")
print("=" * 70)

for policy in POLICIES:
    print(f"\n--- Policy: {policy.value} ---")
    result = wrapper.explain_factual(X_demo, reject_policy=policy)
    _summarize_result(result, label=f"explain_factual(reject_policy={policy.value})")

REJECT POLICY DEMONSTRATION (10 instances: fragile -> robust)

--- Policy: none ---



--- explain_factual(reject_policy=none) ---
type: <class 'calibrated_explanations.explanations.explanations.CalibratedExplanations'>
len: 10
explained_count: 10 / 10

--- Policy: flag ---



--- explain_factual(reject_policy=flag) ---
type: <class 'calibrated_explanations.explanations.reject.RejectCalibratedExplanations'>
policy: RejectPolicy.FLAG
rejected_count: 1 / 10
has_prediction: False
explanation: None

--- Policy: only_rejected ---

--- explain_factual(reject_policy=only_rejected) ---
type: <class 'calibrated_explanations.explanations.reject.RejectCalibratedExplanations'>
policy: RejectPolicy.ONLY_REJECTED
rejected_count: 1 / 10
has_prediction: False
explanation: None

--- Policy: only_accepted ---

--- explain_factual(reject_policy=only_accepted) ---
type: <class 'calibrated_explanations.explanations.reject.RejectCalibratedExplanations'>
policy: RejectPolicy.ONLY_ACCEPTED
rejected_count: 1 / 10
has_prediction: False
explanation: None


In [6]:
# Demonstrate explainer-level default policy
print("\n" + "=" * 70)
print("DEFAULT POLICY DEMONSTRATION")
print("=" * 70)

# Set explainer-level default via calibrate (for WrapCalibratedExplainer)
# Note: We need to re-calibrate to set the default policy
wrapper.calibrate(x_cal, y_cal, default_reject_policy=RejectPolicy.ONLY_REJECTED)
print(f"\nExplainer default: {wrapper.explainer.default_reject_policy}")

result_default = wrapper.explain_factual(X_demo)
_summarize_result(result_default, label="explain_factual() with default ONLY_REJECTED")

# Per-call override takes precedence
result_override = wrapper.explain_factual(X_demo, reject_policy=RejectPolicy.NONE)
_summarize_result(result_override, label="explain_factual(reject_policy=NONE) - override")

# Reset to NONE
wrapper.calibrate(x_cal, y_cal, default_reject_policy=RejectPolicy.NONE)


DEFAULT POLICY DEMONSTRATION

Explainer default: RejectPolicy.ONLY_REJECTED

--- explain_factual() with default ONLY_REJECTED ---
type: <class 'calibrated_explanations.explanations.reject.RejectCalibratedExplanations'>
policy: RejectPolicy.ONLY_REJECTED
rejected_count: 1 / 10
has_prediction: False
explanation: None



--- explain_factual(reject_policy=NONE) - override ---
type: <class 'calibrated_explanations.explanations.explanations.CalibratedExplanations'>
len: 10
explained_count: 10 / 10


WrapCalibratedExplainer(learner=RandomForestClassifier(n_estimators=10, random_state=42), fitted=True, calibrated=True, 
		explainer=CalibratedExplainer(mode=classification, learner=RandomForestClassifier(n_estimators=10, random_state=42)))

In [7]:
# Metadata inspection using our 10 selected instances
print("\n" + "=" * 70)
print("METADATA INSPECTION")
print("=" * 70)

result = wrapper.explain_factual(X_demo, reject_policy=RejectPolicy.FLAG)

print(f"\nPolicy: {result.policy}")
print(f"Rejected array shape: {result.rejected.shape if result.rejected is not None else None}")
print(f"Rejected instances: {np.sum(result.rejected) if result.rejected is not None else 0}")

if result.metadata:
    print("\nMetadata contents:")
    # Print aggregate rates and per-instance masks of interest
    print(f"  reject_rate: {result.metadata.get('reject_rate', 0.0):.4f}")
    print(f"  ambiguity_rate: {result.metadata.get('ambiguity_rate', 0.0):.4f}")
    print(f"  novelty_rate: {result.metadata.get('novelty_rate', 0.0):.4f}")
    amb_mask = result.metadata.get('ambiguity_mask')
    nov_mask = result.metadata.get('novelty_mask')
    sizes = result.metadata.get('prediction_set_size')
    print(f"  ambiguity_mask_count: {int(np.sum(amb_mask)) if amb_mask is not None else 0}")
    print(f"  novelty_mask_count: {int(np.sum(nov_mask)) if nov_mask is not None else 0}")
    print(f"  prediction_set_size sample: {sizes[:10] if sizes is not None else None}")
else:
    print("\nMetadata: None")

# Show which specific instances were rejected
if result.rejected is not None:
    print("\nPer-instance rejection status:")
    for i, (rej, p) in enumerate(zip(result.rejected, proba_demo, strict=False)):
        status = "REJECTED" if rej else "ACCEPTED"
        print(f"  [{i}] P(class=1)={p[1]:.3f} -> {status}")


METADATA INSPECTION



Policy: RejectPolicy.FLAG
Rejected array shape: (10,)
Rejected instances: 1

Metadata contents:
  reject_rate: 0.1000
  ambiguity_rate: 0.1000
  novelty_rate: 0.0000
  ambiguity_mask_count: 1
  novelty_mask_count: 0
  prediction_set_size sample: [2 1 1 1 1 1 1 1 1 1]

Per-instance rejection status:
  [0] P(class=1)=0.517 -> REJECTED
  [1] P(class=1)=0.860 -> ACCEPTED
  [2] P(class=1)=0.860 -> ACCEPTED
  [3] P(class=1)=0.960 -> ACCEPTED
  [4] P(class=1)=0.960 -> ACCEPTED
  [5] P(class=1)=0.960 -> ACCEPTED
  [6] P(class=1)=0.960 -> ACCEPTED
  [7] P(class=1)=0.026 -> ACCEPTED
  [8] P(class=1)=0.026 -> ACCEPTED
  [9] P(class=1)=0.026 -> ACCEPTED


In [8]:
result = wrapper.explain_factual(X_demo, reject_policy=RejectPolicy.ONLY_REJECTED)
print(result.to_narrative(expertise_level='advanced', output_format='text'))


Instance 0

Factual Explanation (Advanced):
--------------------------------------------------------------------------------
⚠️ Use caution: calibrated probability interval is wide (0.657).

REJECTED (ambiguous)
Prediction Set: {'1', '0'} (size: 2)
Confidence: 0.95
Interpretation: The model cannot confidently distinguish between the listed classes at the given confidence level.


Prediction: 1
Calibrated Probability: 0.517
Prediction Interval: [0.200, 0.857]

Factors impacting the calibrated probability for class 1 positively:
1 (15.18) <= 17.34    — weight ≈ 0.392 [0.374, 0.517] [⚠️ highly uncertain]
23 (766.9) <= 1026.50 — weight ≈ 0.184 [0.117, 0.317] [⚠️ highly uncertain]
27 (0.16) > 0.15      — weight ≈ -0.330 [-0.406, -0.316] [⚠️ highly uncertain]
16 (0.03) > 0.02      — weight ≈ -0.203 [-0.340, -0.149] [⚠️ highly uncertain]
26 (0.31) > 0.28      — weight ≈ -0.203 [-0.340, -0.149] [⚠️ highly uncertain]
4 (0.1) > 0.09        — weight ≈ -0.135 [-0.340, 0.006] [⚠️ highly uncertain,

C:\Users\loftuw\Documents\Github\calibrated_explanations\src\calibrated_explanations\explanations\explanations.py:1468: UserWarning: Narrative template fallback: default template used because provided relative path was missing
  return plugin.plot(


## Legacy funcitonality
The legacy `predict_reject` function is still available for backward compatibility, but we recommend using the reject policy system instead for new code. The `predict_reject` function will be deprecated in future releases.